## Phase 3 — Week 1: LLM Fundamentals

### Learning objectives:
- Understand how LLMs work (tokens, context, temperature)
- Make first Groq API call
- Understand system vs user prompts
- Build a fraud alert summarizer using prompt engineering

In [2]:
# ============================================================
# CELL 1: First LLM API Call
# ============================================================

import os
from groq import Groq

# Initialize client — reads API key from environment variable
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Make your first API call
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",  # fast, free LLaMA model
    messages=[
        {
            "role": "system",
            "content": "You are a helpful fraud analyst assistant at a bank."
        },
        {
            "role": "user", 
            "content": "What are the top 3 signs that a credit card transaction might be fraudulent?"
        }
    ],
    temperature=0.3,  # low temperature for consistent factual output
    max_tokens=300    # limit response length
)

# Extract and print the response
print(response.choices[0].message.content)

# Print token usage
print(f"\n--- Token Usage ---")
print(f"Prompt tokens:     {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens:      {response.usage.total_tokens}")

As a fraud analyst assistant at a bank, I've identified the top 3 signs that a credit card transaction might be fraudulent:

1. **Geographic Anomalies**: If a cardholder's usual spending patterns show a sudden and unexpected change in location, it could be a red flag. For example, if a cardholder typically makes purchases in their hometown, but suddenly starts making transactions in a different city or country, it may indicate that the card has been compromised or is being used by an unauthorized party.

2. **Unusual Transaction Amounts or Frequencies**: If a cardholder's transactions show unusual patterns, such as:
	* Large or frequent transactions in a short period.
	* Transactions that exceed the cardholder's usual spending limits.
	* Repeated transactions to the same merchant or location.
	* Transactions that occur during unusual hours (e.g., late at night).

3. **Device or Browser Anomalies**: If a cardholder's transactions show signs of being made from an unfamiliar device or bro

In [8]:
# ============================================================
# CELL 2: Temperature Experiment
# ============================================================

def ask_llm(question, temperature):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful fraud analyst assistant at a bank."
            },
            {
                "role": "user",
                "content": question
            }
        ],
        temperature=temperature,
        max_tokens=150
    )
    return response.choices[0].message.content

question = "In one sentence, what is the biggest sign of credit card fraud?"

print("=== Temperature 0.0 (deterministic) ===")
print(ask_llm(question, 0.0))

print("\n=== Temperature 0.0 (run again - should be identical) ===")
print(ask_llm(question, 0.0))

print("\n=== Temperature 1.0 (creative) ===")
print(ask_llm(question, 1.0))

print("\n=== Temperature 1.0 (run again - will differ) ===")
print(ask_llm(question, 1.0))

=== Temperature 0.0 (deterministic) ===
The biggest sign of credit card fraud is a sudden and unexplained increase in transactions, especially if they are international, online, or involve high-value purchases, which may indicate that the card has been compromised or stolen.

=== Temperature 0.0 (run again - should be identical) ===
The biggest sign of credit card fraud is a sudden and unexplained increase in transactions, especially if they are international, online, or involve high-value purchases, which may indicate that the card has been compromised or stolen.

=== Temperature 1.0 (creative) ===
One of the biggest signs of credit card fraud is a sudden and unusual increase in transaction activity, such as multiple transactions or high-value purchases, which can indicate that your card has been compromised or is being used by an unauthorized person.

=== Temperature 1.0 (run again - will differ) ===
The biggest sign of credit card fraud is an unusual or significant variation in a ca

In [9]:
# ============================================================
# CELL 3: Fraud Alert Summarizer
# ============================================================

def generate_fraud_summary(transaction):
    """
    Takes a transaction dictionary and generates
    a professional fraud investigation summary.
    """
    
    # Format transaction data as structured text
    transaction_text = f"""
    Transaction ID: {transaction['id']}
    Amount: ${transaction['amount']:,.2f}
    Merchant: {transaction['merchant']}
    Location: {transaction['location']}
    Time: {transaction['time']}
    Card Type: {transaction['card_type']}
    Previous avg spend: ${transaction['avg_spend']:,.2f}
    Fraud Score: {transaction['fraud_score']:.2%}
    """
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": """You are a senior fraud analyst at a bank. 
                When given transaction details, you write concise 
                professional investigation summaries.
                
                Your summaries must include:
                1. Risk assessment (High/Medium/Low)
                2. Key red flags identified
                3. Recommended action
                
                Keep summaries under 150 words. Be direct and factual."""
            },
            {
                "role": "user",
                "content": f"Generate a fraud investigation summary for this transaction:\n{transaction_text}"
            }
        ],
        temperature=0.1,
        max_tokens=200
    )
    
    return response.choices[0].message.content

# Test with a suspicious transaction
suspicious_transaction = {
    "id": "TXN-2024-88821",
    "amount": 4500.00,
    "merchant": "Electronics Plus",
    "location": "Lagos, Nigeria",
    "time": "02:47 AM",
    "card_type": "Visa Debit",
    "avg_spend": 85.00,
    "fraud_score": 0.94
}

# Test with a normal transaction
normal_transaction = {
    "id": "TXN-2024-88822",
    "amount": 42.50,
    "merchant": "Starbucks",
    "location": "New York, US",
    "time": "08:15 AM",
    "card_type": "Mastercard Credit",
    "avg_spend": 38.00,
    "fraud_score": 0.03
}

print("=== SUSPICIOUS TRANSACTION ===")
print(generate_fraud_summary(suspicious_transaction))

print("\n=== NORMAL TRANSACTION ===")
print(generate_fraud_summary(normal_transaction))

=== SUSPICIOUS TRANSACTION ===
**Fraud Investigation Summary**

**Transaction ID:** TXN-2024-88821
**Risk Assessment:** High

**Key Red Flags Identified:**

- Excessive transaction amount ($4,500.00) significantly higher than the customer's previous average spend ($85.00).
- High Fraud Score (94.00%).
- Unusual transaction time (02:47 AM).
- Merchant located in Lagos, Nigeria, which is a high-risk country for card-not-present (CNP) transactions.

**Recommended Action:** Immediately flag the transaction for further review and potential reversal. Contact the cardholder to verify the transaction and gather additional information.

=== NORMAL TRANSACTION ===
**Fraud Investigation Summary**

**Transaction ID:** TXN-2024-88822
**Risk Assessment:** Low
**Key Red Flags Identified:** None
**Recommended Action:** No further action required.

The transaction amount of $42.50 is within the customer's average spend of $38.00, indicating a legitimate purchase. The merchant is a reputable Starbucks l